# Uniform Cost Search: Nairobi to Kisumu

The trailer starts at **Nairobi CBD** and must reach **Kisumu Town**. States are towns, actions follow directed road connections, and path cost is the sum of driving distances in kilometres. This notebook implements UCS explicitly and displays its frontier, expansions, final route, and segment costs.

Run all cells from top to bottom. Only Python's standard library is required. The graph is embedded so no live routing service or API key is needed.

## Source and assumptions
The 20 towns and 52 directed distances below are copied from the `edges_with_weights` reconstruction cell in [Part 1 graph.ipynb](https://github.com/i-ssa/PS-Agents/blob/main/graph.ipynb). Reverse directions retain their own costs. Distances describe the supplied abstract graph, not a fresh road survey or a truck-specific routing model. The supplied source already contains a Dijkstra result; the search below is a separate UCS implementation.

Source notebook SHA-256: `ac4d90afae049e474c85a97852f6a5255912c12d74f9530cba8f9c17aec7ec37`.


In [1]:
from heapq import heappush, heappop
from itertools import count
from decimal import Decimal
from math import isfinite

towns = ['Nairobi CBD', 'Ruiru', 'Thika', 'Kiambu', 'Limuru', "Murang'a", 'Sagana', 'Nyeri', 'Nanyuki', 'Nyahururu', 'Naivasha', 'Gilgil', 'Nakuru', 'Eldoret', 'Kericho', 'Kakamega', 'Kisii', 'Ahero', 'Homa Bay', 'Kisumu Town']

edges_with_weights = [('Nairobi CBD', 'Ruiru', 24.78), ('Ruiru', 'Nairobi CBD', 25.83), ('Ruiru', 'Thika', 26.27), ('Thika', 'Ruiru', 24.73), ('Nairobi CBD', 'Kiambu', 40.32), ('Kiambu', 'Nairobi CBD', 40.91), ('Kiambu', 'Limuru', 60.85), ('Limuru', 'Kiambu', 64.25), ('Nairobi CBD', 'Limuru', 32.09), ('Limuru', 'Nairobi CBD', 31.17), ('Thika', "Murang'a", 46.65), ("Murang'a", 'Thika', 44.25), ("Murang'a", 'Sagana', 46.59), ('Sagana', "Murang'a", 46.59), ('Sagana', 'Nyeri', 68.81), ('Nyeri', 'Sagana', 68.71), ('Nyeri', 'Nanyuki', 54.35), ('Nanyuki', 'Nyeri', 54.51), ('Nanyuki', 'Nyahururu', 112.19), ('Nyahururu', 'Nanyuki', 112.02), ('Nyeri', 'Nyahururu', 90.22), ('Nyahururu', 'Nyeri', 90.2), ('Limuru', 'Naivasha', 60.5), ('Naivasha', 'Limuru', 57.32), ('Nairobi CBD', 'Naivasha', 88.45), ('Naivasha', 'Nairobi CBD', 88.49), ('Naivasha', 'Gilgil', 48.18), ('Gilgil', 'Naivasha', 48.39), ('Gilgil', 'Nakuru', 43.78), ('Nakuru', 'Gilgil', 43.12), ('Nyahururu', 'Nakuru', 63.46), ('Nakuru', 'Nyahururu', 62.75), ('Nakuru', 'Eldoret', 152.22), ('Eldoret', 'Nakuru', 154.97), ('Nakuru', 'Kericho', 116.38), ('Kericho', 'Nakuru', 116.58), ('Eldoret', 'Kakamega', 74.59), ('Kakamega', 'Eldoret', 75.08), ('Kakamega', 'Kisumu Town', 87.05), ('Kisumu Town', 'Kakamega', 87.04), ('Kericho', 'Kisumu Town', 72.92), ('Kisumu Town', 'Kericho', 72.98), ('Kericho', 'Kisii', 113.88), ('Kisii', 'Kericho', 113.88), ('Kisii', 'Homa Bay', 72.23), ('Homa Bay', 'Kisii', 72.23), ('Homa Bay', 'Ahero', 106.54), ('Ahero', 'Homa Bay', 106.79), ('Ahero', 'Kisumu Town', 22.44), ('Kisumu Town', 'Ahero', 22.5), ('Kisii', 'Ahero', 99.13), ('Ahero', 'Kisii', 99.32)]

# Store hundredths of a kilometre as integers to avoid floating-point ties.
graph = {town: {} for town in towns}
for origin, destination, km in edges_with_weights:
    graph[origin][destination] = int(Decimal(str(km)) * 100)
assert len(graph) == 20
assert sum(map(len, graph.values())) == 52
print(f"Loaded {len(graph)} towns and 52 directed roads; costs in kilometres.")
for town, roads in graph.items():
    print(town + ': ' + ', '.join(f'{v} ({w/100:.2f})' for v, w in roads.items()))


Loaded 20 towns and 52 directed roads; costs in kilometres.
Nairobi CBD: Ruiru (24.78), Kiambu (40.32), Limuru (32.09), Naivasha (88.45)
Ruiru: Nairobi CBD (25.83), Thika (26.27)
Thika: Ruiru (24.73), Murang'a (46.65)
Kiambu: Nairobi CBD (40.91), Limuru (60.85)
Limuru: Kiambu (64.25), Nairobi CBD (31.17), Naivasha (60.50)
Murang'a: Thika (44.25), Sagana (46.59)
Sagana: Murang'a (46.59), Nyeri (68.81)
Nyeri: Sagana (68.71), Nanyuki (54.35), Nyahururu (90.22)
Nanyuki: Nyeri (54.51), Nyahururu (112.19)
Nyahururu: Nanyuki (112.02), Nyeri (90.20), Nakuru (63.46)
Naivasha: Limuru (57.32), Nairobi CBD (88.49), Gilgil (48.18)
Gilgil: Naivasha (48.39), Nakuru (43.78)
Nakuru: Gilgil (43.12), Nyahururu (62.75), Eldoret (152.22), Kericho (116.38)
Eldoret: Nakuru (154.97), Kakamega (74.59)
Kericho: Nakuru (116.58), Kisumu Town (72.92), Kisii (113.88)
Kakamega: Eldoret (75.08), Kisumu Town (87.05)
Kisii: Kericho (113.88), Homa Bay (72.23), Ahero (99.13)
Ahero: Homa Bay (106.79), Kisumu Town (22.44),

## UCS algorithm
A minimum-priority queue stores cumulative cost, insertion order, and town. Each step removes the cheapest frontier entry. A cheaper discovered route updates the best cost and predecessor; outdated queue entries are skipped. Equal-cost entries use insertion order for deterministic traversal.

The goal is accepted **when removed as the cheapest valid entry**, not when first discovered. With nonnegative edge costs this makes the returned route optimal. The trace records the effective frontier after each expansion; this search order is different from the route the trailer ultimately follows.

In [2]:
def uniform_cost_search(graph, start, goal):
    nodes = set(graph)
    nodes.update(v for roads in graph.values() for v in roads)
    if start not in nodes or goal not in nodes:
        raise ValueError('Start and goal must be graph nodes')
    for roads in graph.values():
        for weight in roads.values():
            if not isfinite(weight) or weight < 0:
                raise ValueError('UCS requires finite nonnegative costs')
    order = count()
    frontier = [(0, next(order), start)]
    best = {start: 0}
    parent = {start: None}
    settled = set()
    trace = []
    while frontier:
        cost, _, town = heappop(frontier)
        if town in settled or cost != best[town]:
            continue
        settled.add(town)
        updates = []
        if town != goal:
            for neighbor, weight in graph.get(town, {}).items():
                candidate = cost + weight
                if neighbor not in settled and candidate < best.get(neighbor, float('inf')):
                    best[neighbor] = candidate
                    parent[neighbor] = town
                    heappush(frontier, (candidate, next(order), neighbor))
                    updates.append((neighbor, candidate))
        active = [(c, i, v) for c, i, v in frontier
                  if v not in settled and c == best[v]]
        trace.append({'town': town, 'cost': cost, 'updates': updates,
                      'frontier': [(v, c) for c, _, v in sorted(active)]})
        if town == goal:
            path = []
            current = goal
            while current is not None:
                path.append(current)
                current = parent[current]
            return path[::-1], cost, trace
    return None, float('inf'), trace


In [3]:
path, total_cost, trace = uniform_cost_search(graph, 'Nairobi CBD', 'Kisumu Town')
for step, row in enumerate(trace, 1):
    print(f"Step {step:02d}: expand {row['town']} | g = {row['cost']/100:.2f} km")
    print('  Added/improved: ' + (', '.join(f'{v}: {c/100:.2f}' for v,c in row['updates']) or 'none'))
    print('  Frontier (cheapest first): ' + (', '.join(f'{v}: {c/100:.2f}' for v,c in row['frontier']) or 'empty'))
print('\nExpansion order: ' + ' -> '.join(row['town'] for row in trace))


Step 01: expand Nairobi CBD | g = 0.00 km
  Added/improved: Ruiru: 24.78, Kiambu: 40.32, Limuru: 32.09, Naivasha: 88.45
  Frontier (cheapest first): Ruiru: 24.78, Limuru: 32.09, Kiambu: 40.32, Naivasha: 88.45
Step 02: expand Ruiru | g = 24.78 km
  Added/improved: Thika: 51.05
  Frontier (cheapest first): Limuru: 32.09, Kiambu: 40.32, Thika: 51.05, Naivasha: 88.45
Step 03: expand Limuru | g = 32.09 km
  Added/improved: none
  Frontier (cheapest first): Kiambu: 40.32, Thika: 51.05, Naivasha: 88.45
Step 04: expand Kiambu | g = 40.32 km
  Added/improved: none
  Frontier (cheapest first): Thika: 51.05, Naivasha: 88.45
Step 05: expand Thika | g = 51.05 km
  Added/improved: Murang'a: 97.70
  Frontier (cheapest first): Naivasha: 88.45, Murang'a: 97.70
Step 06: expand Naivasha | g = 88.45 km
  Added/improved: Gilgil: 136.63
  Frontier (cheapest first): Murang'a: 97.70, Gilgil: 136.63
Step 07: expand Murang'a | g = 97.70 km
  Added/improved: Sagana: 144.29
  Frontier (cheapest first): Gilgil: 13

## Optimal trailer route and cost

In [4]:
if path is None:
    print('No route reaches Kisumu Town.')
else:
    print('Optimal route: ' + ' -> '.join(path))
    print(f'Total driving distance: {total_cost/100:.2f} km\n')
    cumulative = 0
    print(f"{'From':16} {'To':16} {'Segment km':>12} {'Cumulative km':>15}")
    for u, v in zip(path, path[1:]):
        segment = graph[u][v]
        cumulative += segment
        print(f'{u:16} {v:16} {segment/100:12.2f} {cumulative/100:15.2f}')
    assert cumulative == total_cost


Optimal route: Nairobi CBD -> Naivasha -> Gilgil -> Nakuru -> Kericho -> Kisumu Town
Total driving distance: 369.71 km

From             To                 Segment km   Cumulative km
Nairobi CBD      Naivasha                88.45           88.45
Naivasha         Gilgil                  48.18          136.63
Gilgil           Nakuru                  43.78          180.41
Nakuru           Kericho                116.38          296.79
Kericho          Kisumu Town             72.92          369.71


## Verification
Check against an independently implemented Bellman–Ford cost calculation on the same directed graph. Small cases exercise cheaper routes discovered later, unreachable goals, zero-cost cycles, start equal to goal, and invalid negative weights.

In [5]:
def reference_cost(graph, start, goal):
    distances = {v: float('inf') for v in graph}
    distances[start] = 0
    for _ in range(len(graph) - 1):
        changed = False
        for u, roads in graph.items():
            for v, w in roads.items():
                if distances[u] + w < distances[v]:
                    distances[v] = distances[u] + w
                    changed = True
        if not changed:
            break
    return distances[goal]

assert total_cost == reference_cost(graph, 'Nairobi CBD', 'Kisumu Town')
assert total_cost == 36971
assert path == ['Nairobi CBD', 'Naivasha', 'Gilgil', 'Nakuru', 'Kericho', 'Kisumu Town']
assert [r['cost'] for r in trace] == sorted(r['cost'] for r in trace)
example = {'S': {'G': 10, 'A': 1}, 'A': {'G': 1}, 'G': {}}
assert uniform_cost_search(example, 'S', 'G')[:2] == (['S', 'A', 'G'], 2)
assert uniform_cost_search({'S': {}, 'G': {}}, 'S', 'G')[:2] == (None, float('inf'))
assert uniform_cost_search({'S': {}}, 'S', 'S')[:2] == (['S'], 0)
assert uniform_cost_search({'S': {'A': 0}, 'A': {'S': 0, 'G': 1}, 'G': {}}, 'S', 'G')[1] == 1
try:
    uniform_cost_search({'S': {'G': -1}, 'G': {}}, 'S', 'G')
except ValueError:
    pass
else:
    raise AssertionError('Negative edge accepted')
print('All checks passed; independent reference confirms 369.71 km.')


All checks passed; independent reference confirms 369.71 km.


## Interpretation
The minimum-cost route in the supplied graph is **Nairobi CBD → Naivasha → Gilgil → Nakuru → Kericho → Kisumu Town**, with total distance **369.71 km**. UCS explores alternatives in cumulative-cost order before accepting Kisumu. No heuristic is used.

The result is optimal within these 52 directed connections and recorded distances. It does not account for traffic, trailer restrictions, fuel prices, or roads outside the model.